# 01 Audit
Nulls, dtypes, duplicates, key integrity, date ranges and outliers in the raw Olist tables. Source: `src/audit.py` (queries the raw schema in `data/warehouse.duckdb`).

In [1]:
import json, pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
facts = json.load(open("reports/facts.json", encoding="utf-8"))
a = facts["analysis"]
from src import audit
profile, checks = audit.run()

audit: profiled 52 columns across 9 tables; 18 integrity checks


## Integrity checks
Each row is one SQL check against the raw layer. Non-zero counts are handled in the cleaning step.

In [2]:
checks

,check,count
0,duplicate order_id in orders,0
1,"duplicate (order_id, order_item_id) in order_i...",0
2,orders with more than one review,547
3,duplicate review_id rows,814
4,order_items with no matching order,0
5,orders with no order_items,775
6,orders with no payment row,1
7,orders whose customer_id is missing from custo...,0
8,items whose product_id is missing from products,0
9,products with no category,610


## Column profile: columns with missing values

In [3]:
profile[profile.nulls > 0].sort_values('null_pct', ascending=False)

,table,column,dtype,rows,nulls,null_pct,distinct,min,max,iqr_outliers
23,order_reviews,review_comment_title,object,99224,87656,88.34,4527,None,None,NaN
24,order_reviews,review_comment_message,object,99224,58247,58.70,36159,None,None,NaN
6,orders,order_delivered_customer_date,datetime64[us],99441,2965,2.98,95664,2016-10-11 13:46:32,2018-10-17 13:22:46,None
34,products,product_name_lenght,Int32,32951,610,1.85,66,5.0,76.0,290.0
33,products,product_category_name,object,32951,610,1.85,73,NaN,NaN,NaN
35,products,product_description_lenght,Int32,32951,610,1.85,2960,4.0,3992.0,2078.0
36,products,product_photos_qty,Int32,32951,610,1.85,19,1.0,20.0,849.0
5,orders,order_delivered_carrier_date,datetime64[us],99441,1783,1.79,81018,2016-10-08 10:34:01,2018-09-11 19:48:28,None
4,orders,order_approved_at,datetime64[us],99441,160,0.16,90733,2016-09-15 12:16:38,2018-09-03 17:40:06,None
37,products,product_weight_g,float64,32951,2,0.01,2204,0.0,40425.0,4551.0


## Date ranges and numeric outliers

In [4]:
profile[profile['min'].notna()][['table','column','dtype','min','max','iqr_outliers']]

,table,column,dtype,min,max,iqr_outliers
3,orders,order_purchase_timestamp,datetime64[us],2016-09-04 21:15:19,2018-10-17 17:30:18,None
4,orders,order_approved_at,datetime64[us],2016-09-15 12:16:38,2018-09-03 17:40:06,None
5,orders,order_delivered_carrier_date,datetime64[us],2016-10-08 10:34:01,2018-09-11 19:48:28,None
6,orders,order_delivered_customer_date,datetime64[us],2016-10-11 13:46:32,2018-10-17 13:22:46,None
7,orders,order_estimated_delivery_date,datetime64[us],2016-09-30 00:00:00,2018-11-12 00:00:00,None
9,order_items,order_item_id,int32,1,21,13984.0
12,order_items,shipping_limit_date,datetime64[us],2016-09-19 00:15:34,2020-04-09 22:35:08,NaN
13,order_items,price,float64,0.85,6735.0,8427.0
14,order_items,freight_value,float64,0.0,409.68,12134.0
16,order_payments,payment_sequential,int32,1.0,29.0,4526.0


**Read-out.** Duplicate reviews per order, orders without items, products without a category, untranslated categories and partial months at the start and end of the data are the issues that matter; each is resolved in `02_cleaning` with its row impact logged.